In [0]:
%sql

CREATE CATALOG IF NOT EXISTS viagens_teste;

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS viagens_teste.bronze;
CREATE SCHEMA IF NOT EXISTS viagens_teste.silver;
CREATE SCHEMA IF NOT EXISTS viagens_teste.gold;

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS viagens_teste.bronze.arquivos

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/viagens_teste/bronze/arquivos/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/viagens_teste/bronze/arquivos/A_Passagens.csv,A_Passagens.csv,3316,1788178452000
dbfs:/Volumes/viagens_teste/bronze/arquivos/B_Passagens.csv,B_Passagens.csv,3490,1788179040000
dbfs:/Volumes/viagens_teste/bronze/arquivos/Centro_Custo_Map.csv,Centro_Custo_Map.csv,480,1790533518000


In [0]:
df_a = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/A_Passagens.csv",
    header =True,
    inferSchema=True,
    sep=";"
)
display(df_a)

Codigo,Nome Completo,Data Solicitacao,Data Ida,Centro de Custo,Origem,Destino,Valor Total,Situacao
VN-1001,Henrique Lima,2026-10-06,2026-10-11,COSAD,CNF,GIG,"R$ 833,73",Emitida
VN-1002,Bruno Nogueira,2026-09-26,2026-10-21,GPES,GRU,REC,"R$ 531,94",Emitida
VN-1003,Samuel Costa,2026-06-11,2026-06-29,GPES,BSB,CNF,"R$ 1.603,68",Reemitida
VN-1004,Karen Duarte,2026-09-18,2026-09-28,COSAD,CGH,REC,"R$ 1.214,15",Emitida
VN-1005,Lucas Vieira,2026-07-21,2026-07-26,COSAD,SSA,GRU,"R$ 2.780,61",Emitida
VN-1006,Mariana Alves,2026-04-17,2026-05-12,GJUR,GYN,GRU,"R$ 666,70",Emitida
VN-1007,Bruno Nogueira,2026-09-17,2026-10-08,GFIN,GIG,GRU,"R$ 2.268,47",Emitida
VN-1008,Olivia Campos,2026-05-12,2026-05-21,COSAD,SSA,CNF,"R$ 2.198,13",Emitida
VN-1009,Carla Ribeiro,2026-05-29,2026-06-11,GFIN,GIG,CWB,"R$ 2.125,11",Emitida
VN-1010,Mariana Alves,2026-08-20,2026-09-14,GTI,CGH,GYN,"R$ 1.192,36",Emitida


In [0]:
df_a.printSchema()

df_a.columns

root
 |-- Codigo: string (nullable = true)
 |-- Nome Completo: string (nullable = true)
 |-- Data Solicitacao: date (nullable = true)
 |-- Data Ida: date (nullable = true)
 |-- Centro de Custo: string (nullable = true)
 |-- Origem: string (nullable = true)
 |-- Destino: string (nullable = true)
 |-- Valor Total: string (nullable = true)
 |-- Situacao: string (nullable = true)



['Codigo',
 'Nome Completo',
 'Data Solicitacao',
 'Data Ida',
 'Centro de Custo',
 'Origem',
 'Destino',
 'Valor Total',
 'Situacao']

In [0]:
df_b = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/B_Passagens.csv",
    header =True,
    inferSchema=True,
    sep=";"
)
display(df_b)

Nr Reserva,Passageiro,Criado Em,Embarque,CC,De,Para,Tarifa,Status
AT-5001,Vinicius Reis,2026-07-28,2026-08-04,Gestão de Pessoas,GYN,BSB,"R$ 1.341,43",CONFIRMADO
AT-5002,Henrique Lima,2026-05-16,2026-05-29,Financeiro,GIG,BSB,"R$ 2.864,53",CONFIRMADO
AT-5003,Eduarda Melo,2026-04-27,2026-05-22,Jurídico,GRU,GIG,"R$ 2.264,31",ALTERADO
AT-5004,Samuel Costa,2026-06-02,2026-06-12,Gestão de Pessoas,SSA,GIG,"R$ 2.526,98",ALTERADO
AT-5005,Olivia Campos,2026-05-28,2026-06-21,Financeiro,SSA,CWB,"R$ 2.923,94",CONFIRMADO
AT-5006,Ana Martins,2026-05-22,2026-05-31,Ger. Tecnologia,GRU,SSA,"R$ 2.067,85",CONFIRMADO
AT-5007,Henrique Lima,2026-09-19,2026-09-28,Coord. Suprimentos,GRU,BSB,"R$ 635,35",CONFIRMADO
AT-5008,Paulo Mendes,2026-07-20,2026-07-24,Gestão de Pessoas,GIG,CWB,"R$ 1.039,15",CONFIRMADO
AT-5009,Nicolas Souza,2026-09-04,2026-09-24,Jurídico,GIG,GYN,"R$ 973,63",CONFIRMADO
AT-5010,Olivia Campos,2026-10-01,2026-10-16,Financeiro,SSA,REC,"R$ 2.825,55",CONFIRMADO


In [0]:
df_b.printSchema()

df_b.columns

root
 |-- Nr Reserva: string (nullable = true)
 |-- Passageiro: string (nullable = true)
 |-- Criado Em: date (nullable = true)
 |-- Embarque: date (nullable = true)
 |-- CC: string (nullable = true)
 |-- De: string (nullable = true)
 |-- Para: string (nullable = true)
 |-- Tarifa: string (nullable = true)
 |-- Status: string (nullable = true)



['Nr Reserva',
 'Passageiro',
 'Criado Em',
 'Embarque',
 'CC',
 'De',
 'Para',
 'Tarifa',
 'Status']

In [0]:
from pyspark.sql.functions import col, lit

In [0]:
df_a_padrao = df_a.select(
    col("Codigo").alias("id_solicitacao"),
    col("Nome Completo").alias("viajante"),
    col("Data Solicitacao").alias("data_solicitacao"),
    col("Data Ida").alias("data_viagem"),
    col("Centro de Custo").alias("centro_custo_origem"),
    col("Origem").alias("origem"),
    col("Destino").alias("destino"),
    col("Valor Total").alias("valor"),
    col("Situacao").alias("situacao")
).withColumn(
    "agencia",
    lit("ViaNorte")
)

In [0]:
display(df_a_padrao)

id_solicitacao,viajante,data_solicitacao,data_viagem,centro_custo_origem,origem,destino,valor,situacao,agencia
VN-1001,Henrique Lima,2026-10-06,2026-10-11,COSAD,CNF,GIG,"R$ 833,73",Emitida,ViaNorte
VN-1002,Bruno Nogueira,2026-09-26,2026-10-21,GPES,GRU,REC,"R$ 531,94",Emitida,ViaNorte
VN-1003,Samuel Costa,2026-06-11,2026-06-29,GPES,BSB,CNF,"R$ 1.603,68",Reemitida,ViaNorte
VN-1004,Karen Duarte,2026-09-18,2026-09-28,COSAD,CGH,REC,"R$ 1.214,15",Emitida,ViaNorte
VN-1005,Lucas Vieira,2026-07-21,2026-07-26,COSAD,SSA,GRU,"R$ 2.780,61",Emitida,ViaNorte
VN-1006,Mariana Alves,2026-04-17,2026-05-12,GJUR,GYN,GRU,"R$ 666,70",Emitida,ViaNorte
VN-1007,Bruno Nogueira,2026-09-17,2026-10-08,GFIN,GIG,GRU,"R$ 2.268,47",Emitida,ViaNorte
VN-1008,Olivia Campos,2026-05-12,2026-05-21,COSAD,SSA,CNF,"R$ 2.198,13",Emitida,ViaNorte
VN-1009,Carla Ribeiro,2026-05-29,2026-06-11,GFIN,GIG,CWB,"R$ 2.125,11",Emitida,ViaNorte
VN-1010,Mariana Alves,2026-08-20,2026-09-14,GTI,CGH,GYN,"R$ 1.192,36",Emitida,ViaNorte


In [0]:
df_b_padrao = df_b.select(
col("Nr Reserva").alias("id_solicitacao"),
col("Passageiro").alias("viajante"),
col("Criado Em").alias("data_solicitacao"),
col("Embarque").alias("data_viagem"),
col("CC").alias("centro_custo_origem"),
col("De").alias("origem"),
col("Para").alias("destino"),
col("Tarifa").alias("valor"),
col("Status").alias("situacao")
).withColumn(
    "agencia",
    lit("Atlas Corporate")
)
    

In [0]:
display(df_b_padrao)

id_solicitacao,viajante,data_solicitacao,data_viagem,centro_custo_origem,origem,destino,valor,situacao,agencia
AT-5001,Vinicius Reis,2026-07-28,2026-08-04,Gestão de Pessoas,GYN,BSB,"R$ 1.341,43",CONFIRMADO,Atlas Corporate
AT-5002,Henrique Lima,2026-05-16,2026-05-29,Financeiro,GIG,BSB,"R$ 2.864,53",CONFIRMADO,Atlas Corporate
AT-5003,Eduarda Melo,2026-04-27,2026-05-22,Jurídico,GRU,GIG,"R$ 2.264,31",ALTERADO,Atlas Corporate
AT-5004,Samuel Costa,2026-06-02,2026-06-12,Gestão de Pessoas,SSA,GIG,"R$ 2.526,98",ALTERADO,Atlas Corporate
AT-5005,Olivia Campos,2026-05-28,2026-06-21,Financeiro,SSA,CWB,"R$ 2.923,94",CONFIRMADO,Atlas Corporate
AT-5006,Ana Martins,2026-05-22,2026-05-31,Ger. Tecnologia,GRU,SSA,"R$ 2.067,85",CONFIRMADO,Atlas Corporate
AT-5007,Henrique Lima,2026-09-19,2026-09-28,Coord. Suprimentos,GRU,BSB,"R$ 635,35",CONFIRMADO,Atlas Corporate
AT-5008,Paulo Mendes,2026-07-20,2026-07-24,Gestão de Pessoas,GIG,CWB,"R$ 1.039,15",CONFIRMADO,Atlas Corporate
AT-5009,Nicolas Souza,2026-09-04,2026-09-24,Jurídico,GIG,GYN,"R$ 973,63",CONFIRMADO,Atlas Corporate
AT-5010,Olivia Campos,2026-10-01,2026-10-16,Financeiro,SSA,REC,"R$ 2.825,55",CONFIRMADO,Atlas Corporate


In [0]:
df_a_padrao.columns == df_b_padrao.columns

True

In [0]:
df_passagens = df_a_padrao.unionByName(df_b_padrao)

display(df_passagens)

id_solicitacao,viajante,data_solicitacao,data_viagem,centro_custo_origem,origem,destino,valor,situacao,agencia
VN-1001,Henrique Lima,2026-10-06,2026-10-11,COSAD,CNF,GIG,"R$ 833,73",Emitida,ViaNorte
VN-1002,Bruno Nogueira,2026-09-26,2026-10-21,GPES,GRU,REC,"R$ 531,94",Emitida,ViaNorte
VN-1003,Samuel Costa,2026-06-11,2026-06-29,GPES,BSB,CNF,"R$ 1.603,68",Reemitida,ViaNorte
VN-1004,Karen Duarte,2026-09-18,2026-09-28,COSAD,CGH,REC,"R$ 1.214,15",Emitida,ViaNorte
VN-1005,Lucas Vieira,2026-07-21,2026-07-26,COSAD,SSA,GRU,"R$ 2.780,61",Emitida,ViaNorte
VN-1006,Mariana Alves,2026-04-17,2026-05-12,GJUR,GYN,GRU,"R$ 666,70",Emitida,ViaNorte
VN-1007,Bruno Nogueira,2026-09-17,2026-10-08,GFIN,GIG,GRU,"R$ 2.268,47",Emitida,ViaNorte
VN-1008,Olivia Campos,2026-05-12,2026-05-21,COSAD,SSA,CNF,"R$ 2.198,13",Emitida,ViaNorte
VN-1009,Carla Ribeiro,2026-05-29,2026-06-11,GFIN,GIG,CWB,"R$ 2.125,11",Emitida,ViaNorte
VN-1010,Mariana Alves,2026-08-20,2026-09-14,GTI,CGH,GYN,"R$ 1.192,36",Emitida,ViaNorte


In [0]:
display(df_passagens)

id_solicitacao,viajante,data_solicitacao,data_viagem,centro_custo_origem,origem,destino,valor,situacao,agencia
VN-1001,Henrique Lima,2026-10-06,2026-10-11,COSAD,CNF,GIG,"R$ 833,73",Emitida,ViaNorte
VN-1002,Bruno Nogueira,2026-09-26,2026-10-21,GPES,GRU,REC,"R$ 531,94",Emitida,ViaNorte
VN-1003,Samuel Costa,2026-06-11,2026-06-29,GPES,BSB,CNF,"R$ 1.603,68",Reemitida,ViaNorte
VN-1004,Karen Duarte,2026-09-18,2026-09-28,COSAD,CGH,REC,"R$ 1.214,15",Emitida,ViaNorte
VN-1005,Lucas Vieira,2026-07-21,2026-07-26,COSAD,SSA,GRU,"R$ 2.780,61",Emitida,ViaNorte
VN-1006,Mariana Alves,2026-04-17,2026-05-12,GJUR,GYN,GRU,"R$ 666,70",Emitida,ViaNorte
VN-1007,Bruno Nogueira,2026-09-17,2026-10-08,GFIN,GIG,GRU,"R$ 2.268,47",Emitida,ViaNorte
VN-1008,Olivia Campos,2026-05-12,2026-05-21,COSAD,SSA,CNF,"R$ 2.198,13",Emitida,ViaNorte
VN-1009,Carla Ribeiro,2026-05-29,2026-06-11,GFIN,GIG,CWB,"R$ 2.125,11",Emitida,ViaNorte
VN-1010,Mariana Alves,2026-08-20,2026-09-14,GTI,CGH,GYN,"R$ 1.192,36",Emitida,ViaNorte


In [0]:
df_a.count()

199

In [0]:
df_a_padrao = df_a_padrao.filter(
    col("id_solicitacao").isNotNull()
)

df_b_padrao = df_b_padrao.filter(
    col("id_solicitacao").isNotNull()
)

In [0]:
df_b_padrao.count()

18

In [0]:
df_passagens = df_a_padrao.unionByName(df_b_padrao)

In [0]:
print(df_a_padrao.count())
print(df_b_padrao.count())

18
18


In [0]:
print(df_passagens.count())

36


In [0]:
df_passagens.groupBy("agencia").count().show()

+---------------+-----+
|        agencia|count|
+---------------+-----+
|       ViaNorte|   18|
|Atlas Corporate|   18|
+---------------+-----+



In [0]:
df_passagens.select(
    "agencia",
    "valor"
    ).show(20, truncate=False)

df_passagens.printSchema()

+---------------+-----------+
|agencia        |valor      |
+---------------+-----------+
|ViaNorte       |R$ 833,73  |
|ViaNorte       |R$ 531,94  |
|ViaNorte       |R$ 1.603,68|
|ViaNorte       |R$ 1.214,15|
|ViaNorte       |R$ 2.780,61|
|ViaNorte       |R$ 666,70  |
|ViaNorte       |R$ 2.268,47|
|ViaNorte       |R$ 2.198,13|
|ViaNorte       |R$ 2.125,11|
|ViaNorte       |R$ 1.192,36|
|ViaNorte       |R$ 1.553,20|
|ViaNorte       |R$ 1.537,99|
|ViaNorte       |R$ 1.628,19|
|ViaNorte       |R$ 2.528,40|
|ViaNorte       |R$ 1.499,45|
|ViaNorte       |R$ 2.514,91|
|ViaNorte       |R$ 3.072,99|
|ViaNorte       |R$ 2.124,67|
|Atlas Corporate|R$ 1.341,43|
|Atlas Corporate|R$ 2.864,53|
+---------------+-----------+
only showing top 20 rows
root
 |-- id_solicitacao: string (nullable = true)
 |-- viajante: string (nullable = true)
 |-- data_solicitacao: date (nullable = true)
 |-- data_viagem: date (nullable = true)
 |-- centro_custo_origem: string (nullable = true)
 |-- origem: string (nulla

In [0]:
from pyspark.sql.functions import regexp_replace

In [0]:
df_passagens = df_passagens.withColumn(
    "valor_numerico",
    regexp_replace(
        regexp_replace(
            regexp_replace(
                col("valor"),
                "R\\$ ",
                ""
            ),
            "\\.",
            ""
        ),
        ",",
        "."
    ).cast("decimal(18,2)")
)

In [0]:
df_passagens.select(
    "agencia",
    "valor",
    "valor_numerico"
).show(20, truncate=False)

+---------------+-----------+--------------+
|agencia        |valor      |valor_numerico|
+---------------+-----------+--------------+
|ViaNorte       |R$ 833,73  |833.73        |
|ViaNorte       |R$ 531,94  |531.94        |
|ViaNorte       |R$ 1.603,68|1603.68       |
|ViaNorte       |R$ 1.214,15|1214.15       |
|ViaNorte       |R$ 2.780,61|2780.61       |
|ViaNorte       |R$ 666,70  |666.70        |
|ViaNorte       |R$ 2.268,47|2268.47       |
|ViaNorte       |R$ 2.198,13|2198.13       |
|ViaNorte       |R$ 2.125,11|2125.11       |
|ViaNorte       |R$ 1.192,36|1192.36       |
|ViaNorte       |R$ 1.553,20|1553.20       |
|ViaNorte       |R$ 1.537,99|1537.99       |
|ViaNorte       |R$ 1.628,19|1628.19       |
|ViaNorte       |R$ 2.528,40|2528.40       |
|ViaNorte       |R$ 1.499,45|1499.45       |
|ViaNorte       |R$ 2.514,91|2514.91       |
|ViaNorte       |R$ 3.072,99|3072.99       |
|ViaNorte       |R$ 2.124,67|2124.67       |
|Atlas Corporate|R$ 1.341,43|1341.43       |
|Atlas Cor

In [0]:
df_passagens.printSchema()

root
 |-- id_solicitacao: string (nullable = true)
 |-- viajante: string (nullable = true)
 |-- data_solicitacao: date (nullable = true)
 |-- data_viagem: date (nullable = true)
 |-- centro_custo_origem: string (nullable = true)
 |-- origem: string (nullable = true)
 |-- destino: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- situacao: string (nullable = true)
 |-- agencia: string (nullable = false)
 |-- valor_numerico: decimal(18,2) (nullable = true)



In [0]:
from pyspark.sql.functions import sum

df_passagens.select(
    sum("valor_numerico").alias("total_passagens")
).show()

+---------------+
|total_passagens|
+---------------+
|       64664.95|
+---------------+



In [0]:
df_passagens.groupBy("agencia").agg(
    sum("valor_numerico").alias("total_passagens")
).show()

+---------------+---------------+
|        agencia|total_passagens|
+---------------+---------------+
|       ViaNorte|       31874.68|
|Atlas Corporate|       32790.27|
+---------------+---------------+



In [0]:
from pyspark.sql.functions import datediff

In [0]:
df_passagens = df_passagens.withColumn(
    "dias_antecedencia",
    datediff(
        col("data_viagem"),
        col("data_solicitacao")
    )
)

In [0]:
df_passagens.select(
    "id_solicitacao",
    "data_solicitacao",
    "data_viagem",
    "dias_antecedencia"
).show(20, truncate=False)

+--------------+----------------+-----------+-----------------+
|id_solicitacao|data_solicitacao|data_viagem|dias_antecedencia|
+--------------+----------------+-----------+-----------------+
|VN-1001       |2026-10-06      |2026-10-11 |5                |
|VN-1002       |2026-09-26      |2026-10-21 |25               |
|VN-1003       |2026-06-11      |2026-06-29 |18               |
|VN-1004       |2026-09-18      |2026-09-28 |10               |
|VN-1005       |2026-07-21      |2026-07-26 |5                |
|VN-1006       |2026-04-17      |2026-05-12 |25               |
|VN-1007       |2026-09-17      |2026-10-08 |21               |
|VN-1008       |2026-05-12      |2026-05-21 |9                |
|VN-1009       |2026-05-29      |2026-06-11 |13               |
|VN-1010       |2026-08-20      |2026-09-14 |25               |
|VN-1011       |2026-10-11      |2026-10-23 |12               |
|VN-1012       |2026-06-04      |2026-06-24 |20               |
|VN-1013       |2026-05-27      |2026-06

In [0]:
from pyspark.sql.functions import when

In [0]:
df_passagens = df_passagens.withColumn(
    "prazo",
    when(
        col("dias_antecedencia") >= 10,
        "Dentro"
    ).otherwise("Fora")
)

In [0]:
df_passagens.select(
    "id_solicitacao",
    "data_solicitacao",
    "data_viagem",
    "dias_antecedencia",
    "prazo"
).show(20, truncate=False)

+--------------+----------------+-----------+-----------------+------+
|id_solicitacao|data_solicitacao|data_viagem|dias_antecedencia|prazo |
+--------------+----------------+-----------+-----------------+------+
|VN-1001       |2026-10-06      |2026-10-11 |5                |Fora  |
|VN-1002       |2026-09-26      |2026-10-21 |25               |Dentro|
|VN-1003       |2026-06-11      |2026-06-29 |18               |Dentro|
|VN-1004       |2026-09-18      |2026-09-28 |10               |Dentro|
|VN-1005       |2026-07-21      |2026-07-26 |5                |Fora  |
|VN-1006       |2026-04-17      |2026-05-12 |25               |Dentro|
|VN-1007       |2026-09-17      |2026-10-08 |21               |Dentro|
|VN-1008       |2026-05-12      |2026-05-21 |9                |Fora  |
|VN-1009       |2026-05-29      |2026-06-11 |13               |Dentro|
|VN-1010       |2026-08-20      |2026-09-14 |25               |Dentro|
|VN-1011       |2026-10-11      |2026-10-23 |12               |Dentro|
|VN-10

In [0]:
df_passagens.groupBy("prazo").count().show()

+------+-----+
| prazo|count|
+------+-----+
|  Fora|   13|
|Dentro|   23|
+------+-----+



In [0]:
df_passagens.groupBy("prazo").agg(
    sum("valor_numerico").alias("valor_total")
).show()

+------+-----------+
| prazo|valor_total|
+------+-----------+
|  Fora|   24174.55|
|Dentro|   40490.40|
+------+-----------+



In [0]:
df_passagens = df_passagens.withColumn(
    "tipo_despesa",
    lit("Aereo")
)

In [0]:
from pyspark.sql.functions import month

In [0]:
df_passagens = df_passagens.withColumn(
    "trimestre",
    when(
        month(col("data_viagem")).between(1, 3),
        "Primeiro Trimestre"
    )
    .when(
        month(col("data_viagem")).between(4, 6),
        "Segundo Trimestre"
    )
    .when(
        month(col("data_viagem")).between(7, 9),
        "Terceiro Trimestre"
    )
    .otherwise(
        "Quarto Trimestre"
    )
)

In [0]:
df_passagens.select(
    "id_solicitacao",
    "data_viagem",
    "tipo_despesa",
    "trimestre"
).show(20, truncate=False)

+--------------+-----------+------------+------------------+
|id_solicitacao|data_viagem|tipo_despesa|trimestre         |
+--------------+-----------+------------+------------------+
|VN-1001       |2026-10-11 |Aereo       |Quarto Trimestre  |
|VN-1002       |2026-10-21 |Aereo       |Quarto Trimestre  |
|VN-1003       |2026-06-29 |Aereo       |Segundo Trimestre |
|VN-1004       |2026-09-28 |Aereo       |Terceiro Trimestre|
|VN-1005       |2026-07-26 |Aereo       |Terceiro Trimestre|
|VN-1006       |2026-05-12 |Aereo       |Segundo Trimestre |
|VN-1007       |2026-10-08 |Aereo       |Quarto Trimestre  |
|VN-1008       |2026-05-21 |Aereo       |Segundo Trimestre |
|VN-1009       |2026-06-11 |Aereo       |Segundo Trimestre |
|VN-1010       |2026-09-14 |Aereo       |Terceiro Trimestre|
|VN-1011       |2026-10-23 |Aereo       |Quarto Trimestre  |
|VN-1012       |2026-06-24 |Aereo       |Segundo Trimestre |
|VN-1013       |2026-06-06 |Aereo       |Segundo Trimestre |
|VN-1014       |2026-08-

In [0]:
df_passagens.groupBy("trimestre").count().show()

+------------------+-----+
|         trimestre|count|
+------------------+-----+
|  Quarto Trimestre|    6|
| Segundo Trimestre|   16|
|Terceiro Trimestre|   14|
+------------------+-----+



In [0]:
df_map = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/Centro_Custo_Map.csv",
    header=True,
    inferSchema=True,
    sep=";"
)
display(df_map)

Agencia,Valor_Origem,Centro_Custo_Padrao,Diretoria
ViaNorte,COSAD,COSAD,DIRADM
ViaNorte,GTI,GTI,DIRTEC
ViaNorte,GFIN,GFIN,DIRFIN
ViaNorte,GJUR,GJUR,DIRJUR
ViaNorte,GPES,GPES,DIRPES
ViaNorte,GCOM,GCOM,DIRCOM
Atlas Corporate,Coord. Suprimentos,COSAD,DIRADM
Atlas Corporate,Ger. Tecnologia,GTI,DIRTEC
Atlas Corporate,Financeiro,GFIN,DIRFIN
Atlas Corporate,Jurídico,GJUR,DIRJUR


In [0]:
df_map.printSchema()
df_map.columns

root
 |-- Agencia: string (nullable = true)
 |-- Valor_Origem: string (nullable = true)
 |-- Centro_Custo_Padrao: string (nullable = true)
 |-- Diretoria: string (nullable = true)



['Agencia', 'Valor_Origem', 'Centro_Custo_Padrao', 'Diretoria']

In [0]:
df_passagens = (
    df_passagens.alias("p")
    .join(
        df_map.alias("m"),
        (
            (col("p.agencia") == col("m.Agencia")) &
            (col("p.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "p.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

In [0]:
df_passagens = (
    df_passagens.alias("p")
    .join(
        df_map.alias("m"),
        (
            (col("p.agencia") == col("m.Agencia")) &
            (col("p.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "p.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

In [0]:
df_passagens = df_passagens.drop(
    "centro_custo",
    "diretoria"
)

In [0]:
df_passagens.printSchema()

root
 |-- id_solicitacao: string (nullable = true)
 |-- viajante: string (nullable = true)
 |-- data_solicitacao: date (nullable = true)
 |-- data_viagem: date (nullable = true)
 |-- centro_custo_origem: string (nullable = true)
 |-- origem: string (nullable = true)
 |-- destino: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- situacao: string (nullable = true)
 |-- agencia: string (nullable = false)
 |-- valor_numerico: decimal(18,2) (nullable = true)
 |-- dias_antecedencia: integer (nullable = true)
 |-- prazo: string (nullable = false)
 |-- tipo_despesa: string (nullable = false)
 |-- trimestre: string (nullable = false)



In [0]:
df_passagens = (
    df_passagens.alias("p")
    .join(
        df_map.alias("m"),
        (
            (col("p.agencia") == col("m.Agencia")) &
            (col("p.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "p.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

In [0]:
df_passagens.printSchema()

root
 |-- id_solicitacao: string (nullable = true)
 |-- viajante: string (nullable = true)
 |-- data_solicitacao: date (nullable = true)
 |-- data_viagem: date (nullable = true)
 |-- centro_custo_origem: string (nullable = true)
 |-- origem: string (nullable = true)
 |-- destino: string (nullable = true)
 |-- valor: string (nullable = true)
 |-- situacao: string (nullable = true)
 |-- agencia: string (nullable = false)
 |-- valor_numerico: decimal(18,2) (nullable = true)
 |-- dias_antecedencia: integer (nullable = true)
 |-- prazo: string (nullable = false)
 |-- tipo_despesa: string (nullable = false)
 |-- trimestre: string (nullable = false)
 |-- centro_custo: string (nullable = true)
 |-- diretoria: string (nullable = true)



In [0]:
df_passagens.count()

36

In [0]:
df_passagens.filter(
    col("centro_custo").isNull()
).count()

0

In [0]:
df_passagens.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.silver.passagens")

In [0]:
df_teste_leitura = spark.table(
    "viagens_teste.silver.passagens"
)

display(df_teste_leitura)

id_solicitacao,viajante,data_solicitacao,data_viagem,centro_custo_origem,origem,destino,valor,situacao,agencia,valor_numerico,dias_antecedencia,prazo,tipo_despesa,trimestre,centro_custo,diretoria
AT-5001,Vinicius Reis,2026-07-28,2026-08-04,Gestão de Pessoas,GYN,BSB,"R$ 1.341,43",CONFIRMADO,Atlas Corporate,1341.43,7,Fora,Aereo,Terceiro Trimestre,GPES,DIRPES
AT-5002,Henrique Lima,2026-05-16,2026-05-29,Financeiro,GIG,BSB,"R$ 2.864,53",CONFIRMADO,Atlas Corporate,2864.53,13,Dentro,Aereo,Segundo Trimestre,GFIN,DIRFIN
AT-5003,Eduarda Melo,2026-04-27,2026-05-22,Jurídico,GRU,GIG,"R$ 2.264,31",ALTERADO,Atlas Corporate,2264.31,25,Dentro,Aereo,Segundo Trimestre,GJUR,DIRJUR
AT-5004,Samuel Costa,2026-06-02,2026-06-12,Gestão de Pessoas,SSA,GIG,"R$ 2.526,98",ALTERADO,Atlas Corporate,2526.98,10,Dentro,Aereo,Segundo Trimestre,GPES,DIRPES
AT-5005,Olivia Campos,2026-05-28,2026-06-21,Financeiro,SSA,CWB,"R$ 2.923,94",CONFIRMADO,Atlas Corporate,2923.94,24,Dentro,Aereo,Segundo Trimestre,GFIN,DIRFIN
AT-5006,Ana Martins,2026-05-22,2026-05-31,Ger. Tecnologia,GRU,SSA,"R$ 2.067,85",CONFIRMADO,Atlas Corporate,2067.85,9,Fora,Aereo,Segundo Trimestre,GTI,DIRTEC
AT-5007,Henrique Lima,2026-09-19,2026-09-28,Coord. Suprimentos,GRU,BSB,"R$ 635,35",CONFIRMADO,Atlas Corporate,635.35,9,Fora,Aereo,Terceiro Trimestre,COSAD,DIRADM
AT-5008,Paulo Mendes,2026-07-20,2026-07-24,Gestão de Pessoas,GIG,CWB,"R$ 1.039,15",CONFIRMADO,Atlas Corporate,1039.15,4,Fora,Aereo,Terceiro Trimestre,GPES,DIRPES
AT-5009,Nicolas Souza,2026-09-04,2026-09-24,Jurídico,GIG,GYN,"R$ 973,63",CONFIRMADO,Atlas Corporate,973.63,20,Dentro,Aereo,Terceiro Trimestre,GJUR,DIRJUR
AT-5010,Olivia Campos,2026-10-01,2026-10-16,Financeiro,SSA,REC,"R$ 2.825,55",CONFIRMADO,Atlas Corporate,2825.55,15,Dentro,Aereo,Quarto Trimestre,GFIN,DIRFIN
